In [4]:
import joblib
import json
import pandas as pd
from src.data_prep import load_raw, clean_dataset
from sklearn.model_selection import train_test_split

rf = joblib.load("models/random_forest_keylogger_final.joblib")
with open("models/features_keylogger.json") as f:
    features = json.load(f)

df = clean_dataset(load_raw("data/Keylogger_Detection.csv"))
X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_test_fs = X_test[features]

y_test_reset = y_test.reset_index(drop=True)
X_test_reset = X_test_fs.reset_index(drop=True)

indice_teste = y_test_reset[y_test_reset == "Keylogger"].index[0]

print("Tudo pronto:", X_test_reset.shape, "| índice de teste (Keylogger):", indice_teste)

Tudo pronto: (33176, 43) | índice de teste (Keylogger): 4


In [1]:
import joblib

rf = joblib.load("models/random_forest_keylogger_final.joblib")
print("Random Forest ainda carrega bem, apesar do numpy mais antigo.")

from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="qwen2.5:1.5b", base_url="http://localhost:11434")
resposta = llm.invoke("Diz só 'olá' em português, numa frase curta.")
print(resposta.content)

Random Forest ainda carrega bem, apesar do numpy mais antigo.


/Users/lucas/Documents/IAA Ciber Lucas Bacco/IAAC_Grupo2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


"Olá!"


In [2]:
from crewai import Agent, Task, Crew

# Ferramenta simples: vai buscar a previsão exata da Random Forest
def prever_fluxo(indice):
    linha = X_test_reset.iloc[[indice]]
    previsao = rf.predict(linha)[0]
    probas = rf.predict_proba(linha)[0]
    idx_classe = list(rf.classes_).index(previsao)
    confianca = probas[idx_classe]

    top_features = pd.Series(rf.feature_importances_, index=X_test_reset.columns)
    top_features = top_features.sort_values(ascending=False).head(5)
    valores_top = linha[top_features.index].iloc[0]

    dados = "\n".join(f"- {f}: {valores_top[f]:.2f}" for f in top_features.index)
    return f"Classificação: {previsao}\nConfiança: {confianca:.1%}\n\nValores mais relevantes:\n{dados}"

analista = Agent(
    role="Analista de Rede",
    goal="Obter a classificação exata de um fluxo de rede usando o modelo treinado, sem inventar números",
    backstory="És um analista técnico que só reporta dados exatos do modelo de deteção, nunca especula.",
    llm=llm,
    verbose=True,
)

redator = Agent(
    role="Redator de Segurança",
    goal="Escrever relatórios curtos e claros para uma equipa de SOC, em português",
    backstory="És responsável por traduzir dados técnicos em recomendações compreensíveis para analistas de segurança.",
    llm=llm,
    verbose=True,
)

dados_fluxo = prever_fluxo(indice_teste)  # reutiliza o índice de keylogger de antes

tarefa_analise = Task(
    description=f"Reporta exatamente estes dados, sem alterar nenhum número: {dados_fluxo}",
    expected_output="Os dados reportados tal como recebidos, em texto corrido.",
    agent=analista,
)

tarefa_relatorio = Task(
    description="Com base no relatório do Analista de Rede, escreve um relatório final em português (máx. 4 frases) para a equipa de segurança, explicando a situação e recomendando uma ação se for Keylogger.",
    expected_output="Um relatório curto em português.",
    agent=redator,
    context=[tarefa_analise],
)

equipa = Crew(
    agents=[analista, redator],
    tasks=[tarefa_analise, tarefa_relatorio],
    verbose=True,
)

resultado = equipa.kickoff()
print("\n--- RESULTADO FINAL ---")
print(resultado)

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [3]:
class Agente:
    def __init__(self, nome, papel):
        self.nome = nome
        self.papel = papel

    def executar(self, tarefa, contexto=""):
        raise NotImplementedError


class AgenteAnalista(Agente):
    """Agente determinístico: só vai buscar a previsão exata do modelo, sem LLM."""

    def __init__(self, modelo_rf, X_dados):
        super().__init__("Analista de Rede", "Reportar dados exatos do modelo, sem especular")
        self.modelo_rf = modelo_rf
        self.X_dados = X_dados

    def executar(self, indice):
        linha = self.X_dados.iloc[[indice]]
        previsao = self.modelo_rf.predict(linha)[0]
        probas = self.modelo_rf.predict_proba(linha)[0]
        idx_classe = list(self.modelo_rf.classes_).index(previsao)
        confianca = probas[idx_classe]

        top_features = pd.Series(self.modelo_rf.feature_importances_, index=self.X_dados.columns)
        top_features = top_features.sort_values(ascending=False).head(5)
        valores_top = linha[top_features.index].iloc[0]

        dados = "\n".join(f"- {f}: {valores_top[f]:.2f}" for f in top_features.index)
        print(f"[{self.nome}] a reportar dados do fluxo #{indice}...")
        return f"Classificação: {previsao}\nConfiança: {confianca:.1%}\n\nValores mais relevantes:\n{dados}"


class AgenteRedator(Agente):
    """Agente generativo: escreve o relatório final com base nos dados do Analista."""

    def __init__(self, llm):
        super().__init__("Redator de Segurança", "Escrever relatórios claros para a equipa de SOC")
        self.llm = llm

    def executar(self, dados_do_analista):
        prompt = f"""Com base EXATAMENTE nestes dados (não alteres nenhum número), escreve um relatório curto (máx. 4 frases) em português, para uma equipa de segurança:

{dados_do_analista}

Explica a situação em linguagem simples e, se for Keylogger, recomenda uma ação."""
        print(f"[{self.nome}] a escrever o relatório final...")
        resposta = self.llm.invoke(prompt)
        return resposta.content


class EquipaSeguranca:
    def __init__(self, analista, redator):
        self.analista = analista
        self.redator = redator

    def investigar(self, indice):
        dados = self.analista.executar(indice)
        relatorio = self.redator.executar(dados)
        return dados, relatorio


# Montar a equipa
analista = AgenteAnalista(rf, X_test_reset)
redator = AgenteRedator(llm)
equipa = EquipaSeguranca(analista, redator)

dados, relatorio_final = equipa.investigar(indice_teste)

print("\n--- DADOS DO ANALISTA ---")
print(dados)
print("\n--- RELATÓRIO FINAL DO REDATOR ---")
print(relatorio_final)

NameError: name 'X_test_reset' is not defined

In [5]:
class Agente:
    def __init__(self, nome, papel):
        self.nome = nome
        self.papel = papel

    def executar(self, tarefa, contexto=""):
        raise NotImplementedError


class AgenteAnalista(Agente):
    """Agente determinístico: só vai buscar a previsão exata do modelo, sem LLM."""

    def __init__(self, modelo_rf, X_dados):
        super().__init__("Analista de Rede", "Reportar dados exatos do modelo, sem especular")
        self.modelo_rf = modelo_rf
        self.X_dados = X_dados

    def executar(self, indice):
        linha = self.X_dados.iloc[[indice]]
        previsao = self.modelo_rf.predict(linha)[0]
        probas = self.modelo_rf.predict_proba(linha)[0]
        idx_classe = list(self.modelo_rf.classes_).index(previsao)
        confianca = probas[idx_classe]

        top_features = pd.Series(self.modelo_rf.feature_importances_, index=self.X_dados.columns)
        top_features = top_features.sort_values(ascending=False).head(5)
        valores_top = linha[top_features.index].iloc[0]

        dados = "\n".join(f"- {f}: {valores_top[f]:.2f}" for f in top_features.index)
        print(f"[{self.nome}] a reportar dados do fluxo #{indice}...")
        return f"Classificação: {previsao}\nConfiança: {confianca:.1%}\n\nValores mais relevantes:\n{dados}"


class AgenteRedator(Agente):
    """Agente generativo: escreve o relatório final com base nos dados do Analista."""

    def __init__(self, llm):
        super().__init__("Redator de Segurança", "Escrever relatórios claros para a equipa de SOC")
        self.llm = llm

    def executar(self, dados_do_analista):
        prompt = f"""Com base EXATAMENTE nestes dados (não alteres nenhum número), escreve um relatório curto (máx. 4 frases) em português, para uma equipa de segurança:

{dados_do_analista}

Explica a situação em linguagem simples e, se for Keylogger, recomenda uma ação."""
        print(f"[{self.nome}] a escrever o relatório final...")
        resposta = self.llm.invoke(prompt)
        return resposta.content


class EquipaSeguranca:
    def __init__(self, analista, redator):
        self.analista = analista
        self.redator = redator

    def investigar(self, indice):
        dados = self.analista.executar(indice)
        relatorio = self.redator.executar(dados)
        return dados, relatorio


# Montar a equipa
analista = AgenteAnalista(rf, X_test_reset)
redator = AgenteRedator(llm)
equipa = EquipaSeguranca(analista, redator)

dados, relatorio_final = equipa.investigar(indice_teste)

print("\n--- DADOS DO ANALISTA ---")
print(dados)
print("\n--- RELATÓRIO FINAL DO REDATOR ---")
print(relatorio_final)

[Analista de Rede] a reportar dados do fluxo #4...
[Redator de Segurança] a escrever o relatório final...

--- DADOS DO ANALISTA ---
Classificação: Keylogger
Confiança: 87.0%

Valores mais relevantes:
- Source Port: 49202.00
- Flow IAT Min: 8328640.00
- Flow IAT Max: 8328640.00
- Flow Duration: 8328640.00
- Init_Win_bytes_forward: 1727.00

--- RELATÓRIO FINAL DO REDATOR ---
Relatório de Situação: 

A classe de segurança identificou um potencial malware chamado Keylogger com uma classificação de 87%. Essa notificação é um sinal de preocupação, pois isso indica uma grande probabilidade de um software malicioso sendo instalado em sua máquina.

Os valores mais relevantes são os seguintes:
- Source Port: 49202.00
- Flow IAT Min: 8328640.00
- Flow IAT Max: 8328640.00
- Flow Duration: 8328640.00
- Init_Win_bytes_forward: 1727.00

Estes dados sugerem que uma instalação do Keylogger ocorreu com alto êxito, possivelmente através de um escudo DNS em seu sistema, onde os dados foram transferidos d